<a href="https://colab.research.google.com/github/mafloan/Agente-IA-BimBam-Buy/blob/main/Agente_IA_BB_Buy1_Fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# BimBam Buy RAG Agent

This notebook sets up a Retrieval-Augmented Generation (RAG) agent for BimBam Buy policies and documents.

**Fixed Structure:**
1. Install dependencies
2. Define all helper functions
3. Set up APIs and data
4. Create interactive chat interface

In [ ]:
import sys
!{sys.executable} -m pip install -q langchain-community pypdf fpdf2 cohere faiss-cpu langchain-google-genai langchain-text-splitters

print("✅ All dependencies installed successfully!")

## Setup: File Locations and Data

Define where your PDF documents are located.

In [ ]:
import os
from fpdf import FPDF

# Define the directory where the dummy files will be created
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Define your PDF files - UPDATE THESE PATHS TO YOUR ACTUAL PDF LOCATIONS
FILES = [
    (os.path.join(DATA_DIR, "Guia de tiempos y costos.pdf"), "envios"),
    (os.path.join(DATA_DIR, "Manual de Garantía.pdf"), "garantia"),
    (os.path.join(DATA_DIR, "Politica de reembolsos.pdf"), "reembolsos"),
    (os.path.join(DATA_DIR, "Preguntas frecuentes.pdf"), "faq"),
    (os.path.join(DATA_DIR, "Programa de afiliados.pdf"), "afiliados"),
]

# Function to create dummy PDF files for demonstration if they don't exist
def create_dummy_pdf(file_path, content):
    if not os.path.exists(file_path):
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", size=12)
        pdf.multi_cell(0, 10, txt=content)
        pdf.output(file_path)
        print(f"✅ Created dummy PDF: {file_path}")

# Create dummy PDFs for each file in FILES
for path, doc_type in FILES:
    file_name = os.path.basename(path)
    content = f"This is a dummy document for {doc_type}. This file describes policies related to {doc_type}. "  
    content += f"Important information about {doc_type} is contained in this document."
    create_dummy_pdf(path, content)

print(f"\n✅ File setup complete. DATA_DIR: {DATA_DIR}")

## Step 1: Define All Helper Functions

All functions are defined here before being used.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def load_documents():
    """Load all PDF documents from the FILES list."""
    all_docs = []

    for path, doc_type in FILES:
        try:
            loader = PyPDFLoader(path)
            docs = loader.load()

            for d in docs:
                d.metadata["source_doc"] = doc_type
                d.metadata["file_name"] = os.path.basename(path)
                d.metadata["page"] = d.metadata.get("page", 0)

            all_docs.extend(docs)
            print(f"✅ Loaded {len(docs)} pages from '{path}'")

        except Exception as e:
            print(f"❌ Error loading '{path}': {e}")

    return all_docs

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    """Split documents into chunks with overlap for better context preservation."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "]
    )
    chunks = splitter.split_documents(documents)
    print(f"✅ Split {len(documents)} documents into {len(chunks)} chunks")
    return chunks

In [ ]:
from langchain_community.vectorstores import FAISS

def create_vectorstore(chunks, embeddings):
    """Create a FAISS vector store from document chunks."""
    texts = [doc.page_content for doc in chunks]
    metadatas = [doc.metadata for doc in chunks]
    vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metadatas)
    print(f"✅ Created FAISS vectorstore with {len(chunks)} vectors")
    return vectorstore

In [ ]:
from langchain_community.embeddings import CohereEmbeddings

def create_embeddings(api_key):
    """Create Cohere embeddings for semantic search."""
    try:
        embeddings = CohereEmbeddings(
            cohere_api_key=api_key,
            model="embed-multilingual-v3.0",
            user_agent="langchain"
        )
        print("✅ Cohere embeddings initialized successfully")
        return embeddings
    except Exception as e:
        print(f"❌ Error creating embeddings: {e}")
        return None

In [ ]:
def create_retriever(vectorstore):
    """Create a retriever from the vectorstore."""
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    )
    print("✅ Retriever created (k=5 most similar documents)")
    return retriever

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain(retriever, google_api_key):
    """Create a RAG chain with Gemini LLM and document retriever."""
    
    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-pro",
        google_api_key=google_api_key,
        temperature=0.2
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         """Eres un asistente experto en políticas de BimBam Buy.

Usa SOLO la información del contexto proporcionado.

Reglas:
- Responde en español
- Si no sabes la respuesta, di: "No tengo información suficiente"
- Sé claro y directo
- Si aplica, menciona condiciones o excepciones

Contexto:
{context}"""),
        ("human", "{question}")
    ])

    # Build the RAG chain
    rag_chain = (
        RunnableParallel({
            "docs": lambda x: retriever.get_relevant_documents(x["question"]),
            "question": RunnablePassthrough()
        })
        | RunnableParallel({
            "answer": (
                lambda x: {
                    "context": "\n\n".join([d.page_content for d in x["docs"]]),
                    "question": x["question"]
                }
                | prompt
                | llm
                | StrOutputParser()
            ),
            "sources": lambda x: [
                {
                    "doc": d.metadata["source_doc"],
                    "page": d.metadata.get("page", None),
                    "file": d.metadata["file_name"]
                }
                for d in x["docs"]
            ]
        })
    )

    print("✅ RAG chain created successfully")
    return rag_chain

## Step 2: Setup APIs and Initialize Data

In [ ]:
from google.colab import userdata

# Get API keys from Colab Secrets
COHERE_API_KEY = userdata.get("COHERE_API_KEY")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# Validate API keys
if not COHERE_API_KEY:
    raise ValueError("❌ COHERE_API_KEY not found in Colab Secrets")

if not GOOGLE_API_KEY:
    raise ValueError("❌ GOOGLE_API_KEY not found in Colab Secrets")

print("✅ API keys loaded successfully")

In [ ]:
# Step 1: Load documents
print("\n📄 Loading documents...")
docs = load_documents()
print(f"Total documents loaded: {len(docs)}")

# Step 2: Split documents
print("\n✂️  Splitting documents...")
chunks = split_documents(docs)

# Step 3: Create embeddings
print("\n🧠 Creating embeddings...")
embeddings = create_embeddings(COHERE_API_KEY)

if embeddings is None:
    raise RuntimeError("❌ Failed to initialize embeddings")

# Step 4: Create vectorstore
print("\n🔍 Creating vector store...")
vectorstore = create_vectorstore(chunks, embeddings)

# Step 5: Create retriever
print("\n📚 Creating retriever...")
retriever = create_retriever(vectorstore)

# Step 6: Create RAG chain
print("\n⛓️  Creating RAG chain...")
rag_chain = create_rag_chain(retriever, GOOGLE_API_KEY)

print("\n" + "="*50)
print("✅ RAG Pipeline initialized successfully!")
print("="*50)

## Step 3: Interactive Chat Interface

Ask questions about your BimBam Buy policies!

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Create UI components
texto_pregunta = widgets.Text(
    value='',
    placeholder='Ask a question about BimBam Buy policies...',
    description='Question:',
    layout=widgets.Layout(width='70%')
)

boton_enviar = widgets.Button(
    description='Ask Agent',
    button_style='info',
    tooltip='Send query to RAG agent',
    icon='search'
)

salida_respuesta = widgets.Output()

# Event handler for the chat
def procesar_consulta_interactiva(b):
    pregunta = texto_pregunta.value.strip()

    # Validate input
    if not pregunta:
        with salida_respuesta:
            clear_output()
            print("⚠️ Please enter a question before sending.")
        return

    # Disable controls
    texto_pregunta.disabled = True
    boton_enviar.disabled = True

    with salida_respuesta:
        clear_output()
        print("🤖 Processing your question...")
        print("⏳ Retrieving relevant documents and generating response...\n")

        try:
            # Invoke the RAG chain
            resultado = rag_chain.invoke({"question": pregunta})

            # Display results
            clear_output()
            print("🤖 AGENT RESPONSE:")
            print("-" * 60)
            print(resultado["answer"])
            print("-" * 60)

            # Display sources
            print("\n📚 Sources Used:")
            fuentes = set()
            for source_info in resultado["sources"]:
                file_name = source_info.get('file', source_info.get('doc', 'Unknown'))
                fuentes.add(file_name)

            for f in sorted(list(fuentes)):
                print(f"  ✓ {f}")

        except Exception as e:
            clear_output()
            print(f"❌ Error: {str(e)}")
            import traceback
            traceback.print_exc()

    # Re-enable controls
    texto_pregunta.disabled = False
    boton_enviar.disabled = False
    texto_pregunta.value = ''

# Bind button click
boton_enviar.on_click(procesar_consulta_interactiva)

# Display interface
caja_entrada = widgets.HBox([texto_pregunta, boton_enviar])
interfaz_completa = widgets.VBox([caja_entrada, salida_respuesta])

print("🎯 RAG Agent Chat Console Ready!")
print("Ask questions about BimBam Buy policies below:\n")
display(interfaz_completa)

## Test Examples

Try asking questions like:
- "What is the shipping policy?"
- "What are the warranty terms?"
- "How does the refund process work?"
- "Tell me about the affiliate program"
- "What are the frequently asked questions?"

In [ ]:
# Optional: Test with a sample query
print("Testing RAG chain with a sample query...\n")

sample_question = "What are the main policies?"
print(f"Question: {sample_question}")
print("-" * 60)

result = rag_chain.invoke({"question": sample_question})
print("\nResponse:")
print(result["answer"])

print("\n\nSources:")
for source in result["sources"]:
    print(f"  - {source['file']} (Type: {source['doc']})")